# NB02: Update ModelSEEDDatabase with OPAM2 pKa Values

Write OPAM2 pKa predictions into the ModelSEEDDatabase compound TSV files,
replacing the current Marvin 23.4 (ChemAxon) values.

**Format**: ModelSEED uses `fragment:atom:value;...` where atom indices are 1-based.
OPAM2 returns 0-based atom indices, so we add 1.

**Acid dict**: OPAM2 returns H atom indices for acid sites — we map to the heavy-atom
neighbor (same logic as `benchmarks/run_modelseed_comparison.py`).

**Base dict**: OPAM2 returns heavy-atom indices directly for base sites.

In [ ]:
import sys
import glob
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog('rdApp.*')

OPAM2_ROOT = Path('/tmp/OPAM2')
MSDB_ROOT = Path('/tmp/ModelSEEDDatabase')
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'

sys.path.insert(0, str(OPAM2_ROOT / 'src'))
from predict_pka import load_model, model_pred, DEFAULT_ACID_MODEL, DEFAULT_BASE_MODEL
from utils.ionization_group import get_ionization_aid

MOL_DIR = OPAM2_ROOT / 'src' / 'datasets' / 'Unique_MS_Structures'

## 1. Run OPAM2 predictions on all compounds with .mol files

We bypass `predict()` to preserve atom indices (same approach as the benchmark).
Acid predictions: map H atom → heavy-atom neighbor (1-based).
Base predictions: heavy-atom index + 1 (to 1-based).

In [ ]:
acid_model = load_model(DEFAULT_ACID_MODEL)
base_model = load_model(DEFAULT_BASE_MODEL)

mol_files = sorted(MOL_DIR.glob('cpd*.mol'))
print(f'Found {len(mol_files):,} .mol files')

predictions = {}  # cpd_id -> {'pka': str, 'pkb': str}
errors = []

for mol_path in tqdm(mol_files, desc='Predicting pKa'):
    cpd_id = mol_path.stem
    try:
        mol = Chem.MolFromMolFile(str(mol_path))
        if mol is None:
            errors.append((cpd_id, 'MolFromMolFile returned None'))
            continue

        un = rdMolStandardize.Uncharger()
        mol = un.uncharge(mol)
        if mol is None:
            errors.append((cpd_id, 'Uncharger returned None'))
            continue
        mol = AllChem.AddHs(mol)

        # Acid predictions (H atom indices -> heavy atom neighbor, 1-based)
        acid_aids = get_ionization_aid(mol, acid_or_base='acid')
        acid_entries = []
        for h_aid in acid_aids:
            pka = model_pred(mol, h_aid, acid_model)
            h_atom = mol.GetAtomWithIdx(h_aid)
            neighbors = h_atom.GetNeighbors()
            if not neighbors:
                continue
            heavy_idx_1based = neighbors[0].GetIdx() + 1
            acid_entries.append(f'1:{heavy_idx_1based}:{float(pka):.2f}')

        # Base predictions (heavy atom indices, convert to 1-based)
        base_aids = get_ionization_aid(mol, acid_or_base='base')
        base_entries = []
        for b_aid in base_aids:
            pka = model_pred(mol, b_aid, base_model)
            base_entries.append(f'1:{int(b_aid) + 1}:{float(pka):.2f}')

        pka_str = ';'.join(acid_entries) if acid_entries else ''
        pkb_str = ';'.join(base_entries) if base_entries else ''

        if pka_str or pkb_str:
            predictions[cpd_id] = {'pka': pka_str, 'pkb': pkb_str}

    except Exception as e:
        errors.append((cpd_id, str(e)[:100]))

print(f'\nPredictions generated: {len(predictions):,}')
print(f'Errors: {len(errors)}')
if errors:
    print(f'First 5 errors: {errors[:5]}')

## 2. Save predictions for reference

In [ ]:
pred_df = pd.DataFrame([
    {'cpd_id': cpd_id, 'opam2_pka': v['pka'], 'opam2_pkb': v['pkb']}
    for cpd_id, v in predictions.items()
]).sort_values('cpd_id')

pred_df.to_csv(DATA_DIR / 'opam2_predictions.tsv', sep='\t', index=False)
print(f'Saved {len(pred_df):,} predictions to data/opam2_predictions.tsv')
pred_df.head(10)

## 3. Update compound TSV files

In [ ]:
cpd_files = sorted(glob.glob(str(MSDB_ROOT / 'Biochemistry' / 'compound_*.tsv')))
cpd_files = [f for f in cpd_files if 'provenance' not in f]

total_updated = 0
total_new = 0
total_unchanged = 0

for cpd_file in tqdm(cpd_files, desc='Updating TSV files'):
    df = pd.read_csv(cpd_file, sep='\t')
    file_updated = 0

    for idx, row in df.iterrows():
        cpd_id = row['id']
        if cpd_id not in predictions:
            total_unchanged += 1
            continue

        pred = predictions[cpd_id]
        old_pka = str(row.get('pka', '')) if pd.notna(row.get('pka', '')) else ''
        old_pkb = str(row.get('pkb', '')) if pd.notna(row.get('pkb', '')) else ''

        had_existing = bool(old_pka or old_pkb)

        if pred['pka']:
            df.at[idx, 'pka'] = pred['pka']
        if pred['pkb']:
            df.at[idx, 'pkb'] = pred['pkb']

        if had_existing:
            total_updated += 1
        else:
            total_new += 1
        file_updated += 1

    if file_updated > 0:
        df.to_csv(cpd_file, sep='\t', index=False)

print(f'\nTotal compounds updated (replaced Marvin): {total_updated:,}')
print(f'Total compounds with new pKa (had no Marvin): {total_new:,}')
print(f'Total compounds unchanged (no OPAM2 prediction): {total_unchanged:,}')

## 4. Regenerate JSON from updated TSVs

In [ ]:
import subprocess

reprint_script = MSDB_ROOT / 'Scripts' / 'Biochemistry' / 'Reprint_Biochemistry.py'
if reprint_script.exists():
    result = subprocess.run(
        [sys.executable, str(reprint_script)],
        cwd=str(MSDB_ROOT),
        capture_output=True, text=True, timeout=300
    )
    print('STDOUT:', result.stdout[-500:] if result.stdout else '(empty)')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
    else:
        print('JSON regeneration complete.')
else:
    print(f'Reprint script not found at {reprint_script}')
    print('JSON regeneration must be done manually.')

## 5. Verify a few known compounds

In [ ]:
known = {
    'cpd00001': 'Water (pKa ~15.7)',
    'cpd00002': 'ATP',
    'cpd00029': 'Acetate (pKa ~4.76)',
    'cpd00065': 'Tryptophan (amine pKa ~9.39)',
    'cpd00027': 'Glucose',
}

for cpd_id, name in known.items():
    if cpd_id in predictions:
        p = predictions[cpd_id]
        print(f'{cpd_id} ({name}):')
        print(f'  OPAM2 pka: {p["pka"]}')
        print(f'  OPAM2 pkb: {p["pkb"]}')
    else:
        print(f'{cpd_id} ({name}): no OPAM2 prediction')